In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class surface_code():
    def __init__(self):
        np.random.seed(42)
        rng = np.random.default_rng()
        pass

    class check_matrix():
        def __init__(self, dist):
            self.dist = dist
            self.dot_dist = 2*dist-1
            self.num_qubit_data = 2*dist*(dist - 1)+1
            self.num_qubit_measure = self.num_qubit_data - 1

            self.layout = None
    
        # Layout
        def generate_dot_matrix(self, display=False):
            dot_dist = self.dot_dist
            
            r_range = np.arange(dot_dist) 
            c_range = np.arange(dot_dist)
            c_grid, r_grid = np.meshgrid(c_range, r_range)

            r_flat = r_grid.flatten()
            c_flat = c_grid.flatten()
            
            values = np.zeros(dot_dist * dot_dist, dtype=np.int8)
            
            values[r_flat % 2 == 0] = c_flat[r_flat % 2 == 0] % 2
            values[r_flat % 2 == 1] = np.where(c_flat[r_flat % 2 == 1] % 2 == 0, 2, 0)
            
            if display == True:
                x_coords = c_flat
                y_coords = dot_dist - 1 - r_flat  

                mask_0 = (values == 0)
                mask_1 = (values == 1)
                mask_2 = (values == 2)
                
                fig, ax = plt.subplots(figsize=(6, 6))

                for r in range(dot_dist):
                    ax.plot([0, dot_dist - 1], [r, r], color='black', linewidth=1.5, zorder=1)
                for c in range(dot_dist):
                    ax.plot([c, c], [0, dot_dist - 1], color='black', linewidth=1.5, zorder=1)
    
                ax.scatter(x_coords[mask_0], y_coords[mask_0], s=800, marker='o', 
                           facecolor='white', edgecolor = 'black', linewidth=2, zorder=3)
                ax.scatter(x_coords[mask_1], y_coords[mask_1], s=1000, marker='s', 
                           facecolor='lightgreen', linewidth=2, zorder=3)
                ax.scatter(x_coords[mask_1], y_coords[mask_1], s=200, marker='$Z$', color='darkgreen', zorder=4)
                ax.scatter(x_coords[mask_2], y_coords[mask_2], s=1000, marker='s', 
                           facecolor='khaki', linewidth=2, zorder=3)
                ax.scatter(x_coords[mask_2], y_coords[mask_2], s=200, marker='$X$', color='saddlebrown', zorder=4)
            
                ax.set_xticks(c_range)
                ax.set_yticks(r_range)
                ax.set_yticklabels([str(i) for i in range(dot_dist - 1, -1, -1)])
                # ax.grid(True, color='gray', linestyle='--', linewidth=1, zorder=1)
                ax.set_xlim(-0.7, dot_dist - 0.3)
                ax.set_ylim(-0.7, dot_dist - 0.3)
                ax.set_aspect('equal')
                plt.title(f"2D Surface Code Pattern ($d$={self.dist})", fontsize=14, pad=15)
                plt.show()
            
            self.layout = values.reshape(dot_dist, dot_dist)
            pass
    
        # Conversion to check matrices
        def generate_check_matrices(self):
            self.generate_dot_matrix()
            r2, c2 = np.where(self.layout == 2)
            r1, c1 = np.where(self.layout == 1)
            r0, c0 = np.where(self.layout == 0)
    
            check_matrix_Z = np.zeros((self.num_qubit_measure//2, self.num_qubit_data), dtype=np.int8)
            check_matrix_X = np.zeros((self.num_qubit_measure//2, self.num_qubit_data), dtype=np.int8)

            manhattan_dist_Z = np.abs(r1[:, None] - r0) + np.abs(c1[:, None] - c0)
            check_matrix_Z[manhattan_dist_Z == 1] = 1
            manhattan_dist_X = np.abs(r2[:, None] - r0) + np.abs(c2[:, None] - c0)
            check_matrix_X[manhattan_dist_X == 1] = 1
            
            return check_matrix_Z, check_matrix_X

        def simulate_rule1(self, prob):
            check_matrix_Z, check_matrix_X = self.generate_check_matrices()
            
            error_Z = rng.choice(2, self.num_qubit_data, p=[1 - prob/3, prob/3])
            error_X = rng.choice(2, self.num_qubit_data, p=[1 - prob/3, prob/3])

            syndrome_Z = check_matrix_Z @ error_Z % 2
            syndrome_X = check_matrix_X @ error_X % 2

            # We will use PyMatching

In [ ]:
code = surface_code.check_matrix(3)

In [ ]:
code.generate_dot_matrix(display=True)

In [ ]:
code.generate_check_matrices()